In [1]:
#!/usr/bin/env python
"""
Generic audit script for MuProMAC event logs.

- Works for any CSV produced by EventLog / FIFO experiments.
- Does NOT assume a specific scenario; it just inspects stats.
- You interpret the numbers (rework rates, queue behaviour, utilization, etc.).
"""

import os
import glob
import ast
import argparse
from collections import Counter, defaultdict

import numpy as np
import pandas as pd


LANE_KEYS = {"mould_lane", "moulding_lane", "a1_lane", "a2_lane",
             "pack_lane", "sort_lane", "insp_lane"}


# -----------------------------
# Helpers
# -----------------------------

def load_log(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # ensure expected columns exist (handle missing gracefully)
    for col in ["status", "timestamp", "case_id", "activity",
                "resource", "end_time", "cycle_time", "data", "queue_start"]:
        if col not in df.columns:
            df[col] = np.nan
    return df


def parse_data_column(df: pd.DataFrame) -> pd.Series:
    """Parse the 'data' column (stringified dict) into real dicts."""
    def _parse(x):
        if isinstance(x, dict):
            return x
        if not isinstance(x, str) or x.strip() == "":
            return {}
        try:
            return ast.literal_eval(x)
        except Exception:
            return {}
    return df["data"].apply(_parse)


# -----------------------------
# Basic info & sanity checks
# -----------------------------

def basic_summary(df: pd.DataFrame, path: str):
    print("\n" + "=" * 80)
    print(f"FILE: {path}")
    print("=" * 80)

    n_events = len(df)
    n_cases = df["case_id"].nunique()
    activities = df["activity"].dropna().unique()
    resources = df["resource"].dropna().unique()
    statuses = df["status"].dropna().unique()

    print(f"Events      : {n_events}")
    print(f"Cases       : {n_cases}")
    print(f"Activities  : {len(activities)}")
    print(f"Resources   : {len(resources)}")
    print(f"Statuses    : {list(statuses)}")

    # time window
    t_min = df["timestamp"].min()
    t_max = df["timestamp"].max()
    print(f"Time window : [{t_min:.2f}, {t_max:.2f}]  (span={t_max - t_min:.2f})")

    # completion ratio
    completed_cases = df[df["status"] == "COMPLETE"]["case_id"].nunique()
    comp_ratio = completed_cases / n_cases if n_cases > 0 else 0.0
    print(f"Completed cases : {completed_cases} ({comp_ratio:.1%})")

    if comp_ratio < 0.5:
        print("WARNING: <50% of cases completed in horizon → possible overload or too short horizon.")

    # statuses distribution
    print("\nStatus counts:")
    print(df["status"].value_counts().to_string())

    # meta columns
    meta_cols = [c for c in ["scenario", "method", "l", "simulation_run"] if c in df.columns]
    if meta_cols:
        print("\nMeta (unique values):")
        for c in meta_cols:
            print(f"  {c}: {sorted(df[c].dropna().unique())}")


def per_case_monotonicity(df: pd.DataFrame):
    """Check that timestamps are non-decreasing within each case."""
    print("\n=== CASE TIMESTAMP MONOTONICITY ===")
    bad_cases = []
    for cid, grp in df.sort_values(["case_id", "timestamp"]).groupby("case_id"):
        ts = grp["timestamp"].values
        if np.any(np.diff(ts) < -1e-9):
            bad_cases.append(cid)
    if not bad_cases:
        print("All cases have non-decreasing timestamps.")
    else:
        print(f"WARNING: {len(bad_cases)} cases have decreasing timestamps (examples): {bad_cases[:10]}")


# -----------------------------
# Service times & utilization
# -----------------------------

def service_time_stats(df: pd.DataFrame):
    """
    For 'running' events, compute service times = end_time - timestamp
    and summarize per (activity, resource).
    """
    print("\n=== SERVICE TIME STATS (from running events) ===")
    run = df[df["status"] == "running"].copy()
    if run.empty:
        print("No 'running' events found.")
        return

    run["service_time"] = run["end_time"] - run["timestamp"]
    run = run[(run["service_time"] > 0) & np.isfinite(run["service_time"])]

    grp = run.groupby(["activity", "resource"])["service_time"]
    stats = grp.agg(["count", "mean", "std"])
    stats["cv"] = stats["std"] / stats["mean"]
    print(stats.sort_values("mean").head(20).to_string())
    print("\n(Only first 20 rows shown.)")


def resource_utilization(df: pd.DataFrame):
    """
    Approximate utilization and headroom per resource = busy_time / time_span.
    """
    print("\n=== RESOURCE UTILIZATION & HEADROOM (realized) ===")
    run = df[df["status"] == "running"].copy()
    if run.empty:
        print("No 'running' events found.")
        return

    run["service_time"] = run["end_time"] - run["timestamp"]
    run = run[(run["service_time"] > 0) & np.isfinite(run["service_time"])]

    t0 = df["timestamp"].min()
    t1 = df["timestamp"].max()
    horizon = max(t1 - t0, 1e-9)

    util = run.groupby("resource")["service_time"].sum() / horizon
    util = util.sort_values(ascending=False)
    headroom = 1.0 - util

    util_df = pd.DataFrame({
        "utilization": util,
        "headroom": headroom
    })

    print(util_df.to_string())
    print(f"\nMax utilization: {util.max():.3f}, mean: {util.mean():.3f}")
    if util.max() > 0.99:
        print("WARNING: some resources are ~100% busy → near/overload.")
    if (headroom < 0.05).any():
        print("NOTE: very little headroom (<0.05) on some resources.")


# -----------------------------
# Routing / transitions / rework
# -----------------------------

def transition_matrix(df: pd.DataFrame, top_k: int = 20):
    """
    Look at successive activities within each case and estimate transition frequencies.
    """
    print("\n=== ACTIVITY TRANSITIONS (within-case) ===")
    df_sorted = df.sort_values(["case_id", "timestamp"])
    pairs = []

    for cid, grp in df_sorted.groupby("case_id"):
        acts = grp["activity"].tolist()
        for a, b in zip(acts, acts[1:]):
            if pd.isna(a) or pd.isna(b):
                continue
            pairs.append((a, b))

    if not pairs:
        print("No transitions found.")
        return

    cnt = Counter(pairs)
    total = sum(cnt.values())
    print(f"Total observed transitions: {total}")

    print("\nTop transitions:")
    for (a, b), c in cnt.most_common(top_k):
        print(f"{a} → {b}: {c} ({c / total:.2%})")

    # simple rework indicator: self-looping transitions
    self_loops = {k: v for k, v in cnt.items() if k[0] == k[1]}
    if self_loops:
        print("\nSelf-loop (rework) transitions (activity → same activity):")
        total_self = sum(self_loops.values())
        for (a, _), c in sorted(self_loops.items(), key=lambda x: -x[1])[:10]:
            print(f"{a} ↺ {a}: {c} ({c / total:.2%})")
        print(f"Total self-loop transitions: {total_self} ({total_self / total:.2%})")
    else:
        print("\nNo self-loop transitions observed.")


# -----------------------------
# Queue behaviour & FIFO sanity
# -----------------------------

def queue_stability_stats(df: pd.DataFrame, activity_label: str, verbose: bool = True):
    """
    Reconstruct queue length over time for one activity using queued/running events.
    Returns a dict including a 'suspect_overload' flag and slope.
    """
    sub = df[(df["activity"] == activity_label) &
             (df["status"].isin(["queued", "running"]))].copy()
    if sub.empty:
        if verbose:
            print(f"No queue/running events at activity {activity_label}.")
        return {
            "activity": activity_label,
            "has_data": False,
            "max_q": 0,
            "early_mean": 0.0,
            "late_mean": 0.0,
            "ratio": 0.0,
            "slope": 0.0,
            "suspect_overload": False,
        }

    sub = sub.sort_values("timestamp")
    q = 0
    times = []
    sizes = []
    for _, row in sub.iterrows():
        if row["status"] == "queued":
            q += 1
        elif row["status"] == "running":
            q = max(0, q - 1)
        times.append(row["timestamp"])
        sizes.append(q)

    T = times[-1]
    cut_early = T * 0.2
    cut_late = T * 0.8
    early = [s for t, s in zip(times, sizes) if t <= cut_early]
    late = [s for t, s in zip(times, sizes) if t >= cut_late]

    early_mean = float(np.mean(early)) if early else 0.0
    late_mean = float(np.mean(late)) if late else 0.0
    max_q = int(max(sizes)) if sizes else 0

    # linear trend (slope) of queue vs time
    x = np.array(times)
    y = np.array(sizes, dtype=float)
    x_centered = x - x.mean()
    if len(x_centered) > 1:
        slope, intercept = np.polyfit(x_centered, y, 1)
    else:
        slope = 0.0

    eps = 1e-6
    ratio = (late_mean + eps) / (early_mean + eps)

    # crude overload heuristic using both slope and early/late
    suspect_overload = (
        slope > 0.01 and           # upward trend
        late_mean > early_mean + 1.0 and
        ratio > 1.5 and
        late_mean > 2.0            # non-trivial queue
    )

    if verbose:
        print(f"\nQueue @ {activity_label}:")
        print(f"  Final simulated time            : {T:.2f}")
        print(f"  Max queue length                : {max_q}")
        print(f"  Mean queue length early (0–20%) : {early_mean:.2f} (n={len(early)})")
        print(f"  Mean queue length late (80–100%): {late_mean:.2f} (n={len(late)})")
        print(f"  Late / early ratio              : {ratio:.2f}")
        print(f"  Queue length slope (approx)     : {slope:.4f} jobs per time unit")
        if suspect_overload:
            print("  >>> POSSIBLE OVERLOAD: upward trend in queue.")
        else:
            print("  Queue trend not clearly explosive by this heuristic.")

    return {
        "activity": activity_label,
        "has_data": True,
        "max_q": max_q,
        "early_mean": early_mean,
        "late_mean": late_mean,
        "ratio": ratio,
        "slope": slope,
        "suspect_overload": suspect_overload,
    }


def fifo_per_station_check(df: pd.DataFrame):
    """
    Check FIFO per station: at each activity label, tasks should start roughly
    in order of queue_start timestamps.
    """
    print("\n=== FIFO PER STATION CHECK ===")
    qdf = df[df["status"].isin(["queued", "running"]) &
             df["activity"].notna()].copy()
    if qdf.empty:
        print("No queued/running events.")
        return

    # first queue entry per (case_id, activity)
    first_q = (qdf[qdf["status"] == "queued"]
               .sort_values("timestamp")
               .groupby(["case_id", "activity"])
               .agg(queue_start_time=("timestamp", "first"))
               .reset_index())

    # first start per (case_id, activity)
    first_run = (qdf[qdf["status"] == "running"]
                 .sort_values("timestamp")
                 .groupby(["case_id", "activity"])
                 .agg(start_time=("timestamp", "first"))
                 .reset_index())

    merged = pd.merge(first_q, first_run, on=["case_id", "activity"], how="inner")
    if merged.empty:
        print("No matched queued+running pairs.")
        return

    bad_examples = []
    for act, grp in merged.groupby("activity"):
        grp = grp.sort_values("queue_start_time")
        start_times = grp["start_time"].values
        inversions = np.sum(np.diff(start_times) < -1e-9)
        if inversions > 0:
            bad_examples.append((act, inversions, len(grp)))
        print(f"Activity {act}: {len(grp)} starts, inversions={inversions}")

    if bad_examples:
        print("\nActivities with FIFO violations (queue order vs start time):")
        for act, inv, n in bad_examples:
            print(f"  {act}: {inv} inversions over {n} tasks")
    else:
        print("\nNo FIFO order violations detected at activity level (within tolerance).")


# -----------------------------
# Lane keys / stickiness checks
# -----------------------------

def lane_key_stickiness(df: pd.DataFrame, data_parsed: pd.Series):
    """
    For each case, ensure lane keys (mould_lane, a1_lane, etc.)
    do not change over the life of the case (if present).
    """
    print("\n=== LANE KEY STICKINESS (if applicable) ===")
    per_case_keys = defaultdict(lambda: defaultdict(set))

    for (cid, dct) in zip(df["case_id"], data_parsed):
        for k, v in dct.items():
            if k in LANE_KEYS:
                per_case_keys[cid][k].add(v)

    if not per_case_keys:
        print("No lane keys found in data.")
        return

    total_cases = len(per_case_keys)
    bad = []
    for cid, kv in per_case_keys.items():
        for k, vals in kv.items():
            if len(vals) > 1:
                bad.append((cid, k, vals))

    print(f"Cases with any lane key recorded: {total_cases}")
    if not bad:
        print("All lane keys are sticky per case (no key changes detected).")
    else:
        print(f"WARNING: {len(bad)} lane key inconsistencies found (examples):")
        for cid, k, vals in bad[:20]:
            print(f"  case {cid}: {k} had multiple values {vals}")


# -----------------------------
# Main audit entry point
# -----------------------------

def audit_file(path: str):
    df = load_log(path)
    data_parsed = parse_data_column(df)

    # 1) Basic checks
    basic_summary(df, path)
    per_case_monotonicity(df)

    # 2) Service & utilization
    service_time_stats(df)
    resource_utilization(df)

    # 3) Routing / transitions
    transition_matrix(df)

    # 4) Queue stats & FIFO
    print("\n=== QUEUE TRAJECTORIES (top activities by queued events) ===")
    queued_counts = (df[df["status"] == "queued"]["activity"]
                     .value_counts()
                     .head(5))

    overload_flags = []
    for act in queued_counts.index:
        res = queue_stability_stats(df, act, verbose=True)
        if res["has_data"] and res["suspect_overload"]:
            overload_flags.append(res["activity"])

    if overload_flags:
        print("\n>>> GLOBAL WARNING: possible overload detected at activities:")
        for a in overload_flags:
            print(f"    - {a}")
    else:
        print("\nNo strong overload signal in top-queued activities by this heuristic.")

    fifo_per_station_check(df)

    # 5) Lane key stickiness
    lane_key_stickiness(df, data_parsed)

    print("\n=== AUDIT COMPLETE ===\n")


def main():
    # 🔧 HARD-CODE YOUR FOLDER HERE 🔧
    FOLDER = r"out/251110/event_logs/test"

    if not os.path.isdir(FOLDER):
        print(f"Directory not found: {FOLDER}")
        return

    files = sorted(glob.glob(os.path.join(FOLDER, "*.csv")))
    if not files:
        print(f"No CSV files found in directory: {FOLDER}")
        return

    for f in files:
        audit_file(f)


if __name__ == "__main__":
    main()


FILE: out/251110/event_logs/test\log_FIFO_run0_EXP_dedicated_C1.csv
Events      : 265717
Cases       : 8380
Activities  : 51
Resources   : 22
Statuses    : ['START', 'gateway', 'queued', 'running', 'COMPLETE']
Time window : [18.54, 29999.99]  (span=29981.45)
Completed cases : 8368 (99.9%)

Status counts:
status
gateway     110901
queued       69036
running      69032
START         8380
COMPLETE      8368

Meta (unique values):
  method: ['FIFO']
  l: [np.float64(0.0155555555555555)]
  simulation_run: [np.int64(0)]

=== CASE TIMESTAMP MONOTONICITY ===
All cases have non-decreasing timestamps.

=== SERVICE TIME STATS (from running events) ===
                                    count      mean       std        cv
activity       resource                                                
INSPECTION_1_3 INSPECTION_1_3_LINE   4264  1.184357  1.181558  0.997637
INSPECTION_2_3 INSPECTION_2_3_LINE   4353  1.200932  1.209247  1.006924
INSPECTION_1_1 INSPECTION_1_1_LINE   4259  1.203356  1.218823 

In [1]:
#!/usr/bin/env python
"""
Batch visualization for MuProMAC event logs.

For every CSV in FOLDER, this script:
- Prints basic info
- Plots WIP (cases in system) over time
- Plots total queue length over time
- Prints per-station queue statistics (early/late & slopes)

Plots are saved as PNGs next to each CSV file:
- <logname>_wip.png
- <logname>_queue.png
"""

import os
import glob
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# 🔧 CHANGE THIS TO YOUR LOG FOLDER 🔧
FOLDER = r"out/251110/results"


# -----------------------------
# Helpers
# -----------------------------

def load_log(path: str) -> pd.DataFrame:
    # Only load the columns we actually use in this script
    needed = [
        "status",
        "timestamp",
        "case_id",
        "activity",
        "resource",
        "end_time",
        "cycle_time",
        "queue_start",
    ]

    # usecols with a callable: ignore all other columns
    df = pd.read_csv(path, usecols=lambda c: c in needed)

    # ensure expected columns exist (handle missing gracefully)
    for col in needed:
        if col not in df.columns:
            df[col] = np.nan

    return df



def build_step_trajectory(events):
    """
    Generic helper:
    - events: list of (time, delta)
    - returns times, values for a step-like trajectory
    """
    if not events:
        return np.array([]), np.array([])

    events_sorted = sorted(events, key=lambda x: x[0])
    times = []
    values = []
    current = 0.0

    for t, delta in events_sorted:
        current += delta
        times.append(float(t))
        values.append(current)

    return np.array(times), np.array(values)


def build_wip_trajectory(df: pd.DataFrame):
    """
    WIP(t) = number of cases that have started but not yet completed.
    Uses START and COMPLETE events.
    """
    starts = df[df["status"] == "START"]["timestamp"].values
    comps = df[df["status"] == "COMPLETE"]["timestamp"].values

    events = []
    for t in starts:
        events.append((t, +1))
    for t in comps:
        events.append((t, -1))

    return build_step_trajectory(events)


def build_global_queue_trajectory(df: pd.DataFrame):
    """
    Total queue length = number of jobs with status 'queued' not yet
    started ('running') anywhere, aggregated over all activities.
    """
    qs = df[df["status"] == "queued"]["timestamp"].values
    rs = df[df["status"] == "running"]["timestamp"].values

    events = []
    for t in qs:
        events.append((t, +1))
    for t in rs:
        events.append((t, -1))

    return build_step_trajectory(events)


def summarize_step_trajectory(times: np.ndarray, values: np.ndarray, label: str):
    if times.size == 0:
        print(f"{label}: no data.")
        return

    T = times[-1]
    v_max = float(values.max())
    cut_early = 0.2 * T
    cut_late = 0.8 * T

    early_vals = [v for t, v in zip(times, values) if t <= cut_early]
    late_vals = [v for t, v in zip(times, values) if t >= cut_late]

    early_mean = float(np.mean(early_vals)) if early_vals else 0.0
    late_mean = float(np.mean(late_vals)) if late_vals else 0.0
    eps = 1e-6
    ratio = (late_mean + eps) / (early_mean + eps)

    # rough slope (trend over time)
    x = times - times.mean()
    y = values.astype(float)
    if len(x) > 1:
        slope, _ = np.polyfit(x, y, 1)
    else:
        slope = 0.0

    print(f"{label}:")
    print(f"  Horizon T          : {T:.2f}")
    print(f"  Max value          : {v_max:.2f}")
    print(f"  Mean early (0–20%) : {early_mean:.2f}")
    print(f"  Mean late (80–100%): {late_mean:.2f}")
    print(f"  Late / early ratio : {ratio:.2f}")
    print(f"  Slope (approx)     : {slope:.4f}")
    print()


# -----------------------------
# Queue behaviour per station
# -----------------------------

def queue_stability_stats(df: pd.DataFrame, activity_label: str):
    """
    Reconstruct queue length over time for one activity using queued/running events.
    Returns a dict including a 'suspect_overload' flag and slope.
    """
    sub = df[(df["activity"] == activity_label) &
             (df["status"].isin(["queued", "running"]))].copy()
    if sub.empty:
        return {
            "activity": activity_label,
            "has_data": False,
            "max_q": 0,
            "early_mean": 0.0,
            "late_mean": 0.0,
            "ratio": 0.0,
            "slope": 0.0,
            "suspect_overload": False,
        }

    sub = sub.sort_values("timestamp")
    q = 0
    times = []
    sizes = []
    for _, row in sub.iterrows():
        if row["status"] == "queued":
            q += 1
        elif row["status"] == "running":
            q = max(0, q - 1)
        times.append(row["timestamp"])
        sizes.append(q)

    times = np.array(times, dtype=float)
    sizes = np.array(sizes, dtype=float)

    T = times[-1]
    cut_early = T * 0.2
    cut_late = T * 0.8
    early = [s for t, s in zip(times, sizes) if t <= cut_early]
    late = [s for t, s in zip(times, sizes) if t >= cut_late]

    early_mean = float(np.mean(early)) if early else 0.0
    late_mean = float(np.mean(late)) if late else 0.0
    max_q = int(max(sizes)) if sizes.size > 0 else 0

    # linear trend (slope) of queue vs time
    x = times - times.mean()
    y = sizes
    if len(x) > 1:
        slope, _ = np.polyfit(x, y, 1)
    else:
        slope = 0.0

    eps = 1e-6
    ratio = (late_mean + eps) / (early_mean + eps)

    suspect_overload = (
        slope > 0.01 and
        late_mean > early_mean + 1.0 and
        ratio > 1.5 and
        late_mean > 2.0
    )

    return {
        "activity": activity_label,
        "has_data": True,
        "max_q": max_q,
        "early_mean": early_mean,
        "late_mean": late_mean,
        "ratio": ratio,
        "slope": slope,
        "suspect_overload": suspect_overload,
    }


def all_station_queue_summary(df: pd.DataFrame):
    """
    Compute queue stability stats for all activities that ever have 'queued' events.
    """
    queued_acts = df[df["status"] == "queued"]["activity"].dropna().unique()
    results = []

    for act in queued_acts:
        res = queue_stability_stats(df, act)
        if res["has_data"]:
            results.append(res)

    if not results:
        print("No queue data per station.")
        return

    summary = pd.DataFrame(results)
    summary = summary.sort_values("ratio", ascending=False)

    print("=== ALL-STATION QUEUE SUMMARY (sorted by late/early ratio) ===")
    cols = ["activity", "max_q", "early_mean", "late_mean", "ratio", "slope", "suspect_overload"]
    print(summary[cols].to_string(index=False))
    print()


# -----------------------------
# Main visualization for one log
# -----------------------------

def visualize_log(path: str):
    print("\n" + "=" * 80)
    print(f"LOG: {path}")
    print("=" * 80)

    df = load_log(path)

    n_events = len(df)
    n_cases = df["case_id"].nunique()
    t_min = df["timestamp"].min()
    t_max = df["timestamp"].max()

    print(f"Events  : {n_events}")
    print(f"Cases   : {n_cases}")
    print(f"Horizon : [{t_min:.2f}, {t_max:.2f}] span={t_max - t_min:.2f}")

    # WIP trajectory
    wip_t, wip_v = build_wip_trajectory(df)
    summarize_step_trajectory(wip_t, wip_v, "WIP (cases in system)")

    # Global queue trajectory
    q_t, q_v = build_global_queue_trajectory(df)
    summarize_step_trajectory(q_t, q_v, "Total queue length")

    # Save plots next to the CSV
    base_dir = os.path.dirname(path)
    base_name = os.path.splitext(os.path.basename(path))[0]

    # WIP plot
    if wip_t.size > 0:
        plt.figure(figsize=(8, 4))
        plt.step(wip_t, wip_v, where="post")
        plt.xlabel("time")
        plt.ylabel("WIP (cases in system)")
        plt.title(f"WIP over time – {base_name}")
        plt.tight_layout()
        out_path = os.path.join(base_dir, base_name + "_wip.png")
        plt.savefig(out_path, dpi=150)
        plt.close()
        print(f"WIP plot saved to: {out_path}")
    else:
        print("No WIP data to plot.")

    # Queue plot
    if q_t.size > 0:
        plt.figure(figsize=(8, 4))
        plt.step(q_t, q_v, where="post")
        plt.xlabel("time")
        plt.ylabel("Total queue length")
        plt.title(f"Total queue over time – {base_name}")
        plt.tight_layout()
        out_path = os.path.join(base_dir, base_name + "_queue.png")
        plt.savefig(out_path, dpi=150)
        plt.close()
        print(f"Queue plot saved to: {out_path}")
    else:
        print("No queue data to plot.")

    # Per-station queue summary
    all_station_queue_summary(df)

    print("=== VISUALIZATION DONE ===\n")


# -----------------------------
# Batch loop over all CSV logs
# -----------------------------

def main():
    if not os.path.isdir(FOLDER):
        print(f"Directory not found: {FOLDER}")
        return

    files = sorted(glob.glob(os.path.join(FOLDER, "FIFO_EXP_*.csv")))
    if not files:
        print(f"No CSV files found in directory: {FOLDER}")
        return

    print(f"Found {len(files)} CSV log(s) in {FOLDER}.\n")

    for path in files:
        visualize_log(path)


if __name__ == "__main__":
    main()


Found 69 CSV log(s) in out/251110/results.


LOG: out/251110/results\FIFO_EXP_l0.28_dedicated_C1_V18_A5_QC97_identical.csv
Events  : 180945
Cases   : 8461
Horizon : [18.54, 29998.54] span=29980.00
WIP (cases in system):
  Horizon T          : 29998.54
  Max value          : 62.00
  Mean early (0–20%) : 34.43
  Mean late (80–100%): 24.92
  Late / early ratio : 0.72
  Slope (approx)     : -0.0004

Total queue length:
  Horizon T          : 29998.54
  Max value          : 55.00
  Mean early (0–20%) : 27.33
  Mean late (80–100%): 17.74
  Late / early ratio : 0.65
  Slope (approx)     : -0.0004

WIP plot saved to: out/251110/results\FIFO_EXP_l0.28_dedicated_C1_V18_A5_QC97_identical_wip.png
Queue plot saved to: out/251110/results\FIFO_EXP_l0.28_dedicated_C1_V18_A5_QC97_identical_queue.png
=== ALL-STATION QUEUE SUMMARY (sorted by late/early ratio) ===
    activity  max_q  early_mean  late_mean    ratio         slope  suspect_overload
ASSEMBLY_1_1      9    1.677285   1.978947 1.179851  7.0040

In [3]:
#!/usr/bin/env python

"""
Trim warm-up and split MuProMAC event logs into train/val/test.

- Warm-up: drop first 10% of the *time horizon*.
- Split: on the remaining part, split by cases:
    * first 60% of cases  -> train
    * next 20% of cases   -> val
    * last 20% of cases   -> test

One set of 3 CSVs is produced per input log.
"""

import os
import glob

import numpy as np
import pandas as pd


# ------------------------------------------------------------------
# CONFIG: change these to your paths
# ------------------------------------------------------------------
INPUT_FOLDER  = r"out/251110/event_logs/test"
OUTPUT_FOLDER = r"out/251110/event_logs_splits"
WARMUP_FRAC   = 0.10   # drop first 10% of time
TRAIN_FRAC    = 0.60   # of remaining cases
VAL_FRAC      = 0.20   # test gets the rest
# ------------------------------------------------------------------


def load_log(path: str) -> pd.DataFrame:
    """Load a log and make sure timestamp and case_id exist."""
    df = pd.read_csv(path)

    # basic sanity
    for col in ["timestamp", "case_id", "status"]:   # >>> NEW: also require status
        if col not in df.columns:
            raise ValueError(f"{path} is missing required column: {col}")

    return df


def trim_warmup(df: pd.DataFrame, warmup_frac: float) -> pd.DataFrame:
    """Drop the first warmup_frac of the time horizon."""
    t0 = df["timestamp"].min()
    t1 = df["timestamp"].max()
    cut = t0 + warmup_frac * (t1 - t0)

    trimmed = df[df["timestamp"] >= cut].copy()
    trimmed = trimmed.sort_values(["timestamp", "case_id"]).reset_index(drop=True)

    return trimmed


# >>> NEW: keep only complete cases
def keep_complete_cases(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep only cases that have at least one COMPLETE event
    (after warm-up trimming).
    """
    complete_ids = df.loc[df["status"] == "COMPLETE", "case_id"].unique()
    kept = df[df["case_id"].isin(complete_ids)].copy()
    return kept
# <<< END NEW


def split_by_cases(df: pd.DataFrame,
                   train_frac: float,
                   val_frac: float):
    """
    Split cases into train/val/test based on their first timestamp
    (so earlier-entering cases go to train, etc.).
    """
    # When does each case first appear (after warm-up)?
    first_times = (
        df.groupby("case_id")["timestamp"]
          .min()
          .sort_values()
    )

    n_cases = len(first_times)
    if n_cases == 0:
        raise ValueError("No cases left after warm-up trimming (and completeness filter).")

    n_train = int(np.floor(train_frac * n_cases))
    n_val   = int(np.floor(val_frac * n_cases))
    # test gets the remainder
    n_test  = n_cases - n_train - n_val

    train_cases = first_times.index[:n_train]
    val_cases   = first_times.index[n_train:n_train + n_val]
    test_cases  = first_times.index[n_train + n_val:]

    train_df = df[df["case_id"].isin(train_cases)].copy()
    val_df   = df[df["case_id"].isin(val_cases)].copy()
    test_df  = df[df["case_id"].isin(test_cases)].copy()

    return train_df, val_df, test_df, n_train, n_val, n_test


def process_log(path: str):
    base = os.path.splitext(os.path.basename(path))[0]
    print(f"\n=== Processing {base} ===")

    df = load_log(path)
    print(f"  Original events: {len(df)}, cases: {df['case_id'].nunique()}")

    # 1) trim warm-up
    trimmed = trim_warmup(df, WARMUP_FRAC)
    print(f"  After warm-up trim ({WARMUP_FRAC:.0%} time): "
          f"{len(trimmed)} events, {trimmed['case_id'].nunique()} cases")

    # 1b) >>> NEW: keep only complete cases
    trimmed = keep_complete_cases(trimmed)
    print(f"  After dropping incomplete cases: "
          f"{len(trimmed)} events, {trimmed['case_id'].nunique()} cases")

    # 2) split by cases
    train_df, val_df, test_df, n_train, n_val, n_test = split_by_cases(
        trimmed, TRAIN_FRAC, VAL_FRAC
    )
    print(f"  Split by cases -> train: {n_train}, val: {n_val}, test: {n_test}")

    # 3) save
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    train_path = os.path.join(OUTPUT_FOLDER, f"{base}_warm10_train.csv")
    val_path   = os.path.join(OUTPUT_FOLDER, f"{base}_warm10_val.csv")
    test_path  = os.path.join(OUTPUT_FOLDER, f"{base}_warm10_test.csv")

    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)
    test_df.to_csv(test_path, index=False)

    print(f"  Saved:\n"
          f"    {train_path}  ({len(train_df)} events)\n"
          f"    {val_path}    ({len(val_df)} events)\n"
          f"    {test_path}   ({len(test_df)} events)")


def main():
    if not os.path.isdir(INPUT_FOLDER):
        raise SystemExit(f"Input folder not found: {INPUT_FOLDER}")

    files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.csv")))
    if not files:
        raise SystemExit(f"No CSV files in {INPUT_FOLDER}")

    print(f"Found {len(files)} log(s) to process.")
    for path in files:
        # optional: skip already-split files if you run script twice
        if any(suffix in os.path.basename(path)
               for suffix in ["_train", "_val", "_test"]):
            continue
        process_log(path)


if __name__ == "__main__":
    main()


Found 8 log(s) to process.

=== Processing log_FIFO_run0_EXP_dedicated_C1 ===
  Original events: 265717, cases: 8380
  After warm-up trim (10% time): 238959 events, 7552 cases
  After dropping incomplete cases: 238791 events, 7540 cases
  Split by cases -> train: 4524, val: 1508, test: 1508
  Saved:
    out/251110/event_logs_splits\log_FIFO_run0_EXP_dedicated_C1_warm10_train.csv  (143021 events)
    out/251110/event_logs_splits\log_FIFO_run0_EXP_dedicated_C1_warm10_val.csv    (47885 events)
    out/251110/event_logs_splits\log_FIFO_run0_EXP_dedicated_C1_warm10_test.csv   (47885 events)

=== Processing log_FIFO_run0_EXP_hybrid30_C1 ===
  Original events: 231175, cases: 8345
  After warm-up trim (10% time): 207949 events, 7513 cases
  After dropping incomplete cases: 207840 events, 7500 cases
  Split by cases -> train: 4500, val: 1500, test: 1500
  Saved:
    out/251110/event_logs_splits\log_FIFO_run0_EXP_hybrid30_C1_warm10_train.csv  (124668 events)
    out/251110/event_logs_splits\log_

In [11]:
#!/usr/bin/env python

"""
Build prefix-level datasets for remaining-time prediction from MuProMAC event logs.

For each input CSV (already warm-up trimmed and split into train/val/test):
- Keep only *complete* cases (should already be true, but we double-check).
- For each event in a case, compute:
    * prefix_index       : event position within the case
    * elapsed_time       : timestamp - case_start_time
    * remaining_time     : case_complete_time - timestamp

Output: one CSV per input file with prefix-level rows.
"""

import os
import glob

import numpy as np
import pandas as pd


# ------------------------------------------------------------------
# CONFIG: change paths as needed
# ------------------------------------------------------------------
INPUT_FOLDER  = r"out/251110/event_logs_splits"   # your split logs
OUTPUT_FOLDER = r"out/251110/prefix_datasets"     # new folder for prefix datasets
# ------------------------------------------------------------------


def load_split_log(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    # basic sanity
    for col in ["timestamp", "case_id", "status"]:
        if col not in df.columns:
            raise ValueError(f"{path} is missing required column: {col}")

    return df


def keep_complete_cases(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter to cases that:
    - have at least one COMPLETE event
    - and whose *first* event in the (already warm-up trimmed) log is START.
    (This matches the splitting script you used.)
    """
    complete_cases = set(df.loc[df["status"] == "COMPLETE", "case_id"].unique())

    first_events = (
        df.sort_values(["case_id", "timestamp"])
          .groupby("case_id")
          .head(1)
    )
    good_starts = set(
        first_events.loc[first_events["status"] == "START", "case_id"].unique()
    )

    valid_cases = complete_cases & good_starts
    return df[df["case_id"].isin(valid_cases)].copy()


def build_prefix_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    From an event log (only complete cases), build prefix-level data with remaining_time.
    """

    # --- per-case times ---
    case_start = (
        df.groupby("case_id")["timestamp"]
          .min()
    )

    # we assume COMPLETE events exist; take their timestamp as completion time
    comp_times = (
        df[df["status"] == "COMPLETE"]
        .groupby("case_id")["timestamp"]
        .min()
    )

    # keep only cases that have both start and completion
    valid_cases = set(case_start.index) & set(comp_times.index)
    df = df[df["case_id"].isin(valid_cases)].copy()

    # map times back
    df["case_start_time"]     = df["case_id"].map(case_start)
    df["case_complete_time"]  = df["case_id"].map(comp_times)

    # sort within cases
    df = df.sort_values(["case_id", "timestamp"]).reset_index(drop=True)

    # prefix index within case (1,2,3,...)
    df["prefix_index"] = df.groupby("case_id").cumcount() + 1

    # elapsed time since case start
    df["elapsed_time"] = df["timestamp"] - df["case_start_time"]

    # remaining time until completion
    df["remaining_time"] = df["case_complete_time"] - df["timestamp"]

    # optional: drop gateway-only rows if you don’t want them
    # (for now we keep everything; you can filter later if needed)
    # df = df[df["status"] != "gateway"].copy()

    # Choose columns to keep (you can add/remove later)
    keep_cols = [
        "case_id",
        "timestamp",
        "activity",
        "resource",
        "status",
        "prefix_index",
        "elapsed_time",
        "remaining_time",
        "case_start_time",
        "case_complete_time",
    ]

    # plus any meta columns if present
    for extra in ["scenario", "method", "l", "simulation_run", "process"]:
        if extra in df.columns:
            keep_cols.append(extra)

    return df[keep_cols]


def process_split_file(path: str):
    base = os.path.splitext(os.path.basename(path))[0]
    print(f"\n=== Building prefixes for {base} ===")

    df = load_split_log(path)
    print(f"  Input events: {len(df)}, cases: {df['case_id'].nunique()}")

    df = keep_complete_cases(df)
    print(f"  After complete-case filter: {len(df)} events, "
          f"{df['case_id'].nunique()} cases")

    prefix_df = build_prefix_dataset(df)
    print(f"  Prefix rows: {len(prefix_df)} "
          f"(cases: {prefix_df['case_id'].nunique()})")

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    out_path = os.path.join(OUTPUT_FOLDER, f"{base}_prefix.csv")
    prefix_df.to_csv(out_path, index=False)
    print(f"  Saved prefix dataset -> {out_path}")


def main():
    if not os.path.isdir(INPUT_FOLDER):
        raise SystemExit(f"Input folder not found: {INPUT_FOLDER}")

    files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.csv")))
    if not files:
        raise SystemExit(f"No CSV files in {INPUT_FOLDER}")

    print(f"Found {len(files)} split log(s) to process.")
    for path in files:
        # skip already processed prefix files if you re-run
        if path.endswith("_prefix.csv"):
            continue
        process_split_file(path)


if __name__ == "__main__":
    main()


Found 24 split log(s) to process.

=== Building prefixes for log_FIFO_run0_EXP_dedicated_C1_warm10_test ===
  Input events: 47885, cases: 1508
  After complete-case filter: 47885 events, 1508 cases
  Prefix rows: 47885 (cases: 1508)
  Saved prefix dataset -> out/251110/prefix_datasets\log_FIFO_run0_EXP_dedicated_C1_warm10_test_prefix.csv

=== Building prefixes for log_FIFO_run0_EXP_dedicated_C1_warm10_train ===
  Input events: 143021, cases: 4524
  After complete-case filter: 142404 events, 4491 cases
  Prefix rows: 142404 (cases: 4491)
  Saved prefix dataset -> out/251110/prefix_datasets\log_FIFO_run0_EXP_dedicated_C1_warm10_train_prefix.csv

=== Building prefixes for log_FIFO_run0_EXP_dedicated_C1_warm10_val ===
  Input events: 47885, cases: 1508
  After complete-case filter: 47885 events, 1508 cases
  Prefix rows: 47885 (cases: 1508)
  Saved prefix dataset -> out/251110/prefix_datasets\log_FIFO_run0_EXP_dedicated_C1_warm10_val_prefix.csv

=== Building prefixes for log_FIFO_run0_EXP_

In [7]:
!pip install statsforecast


In [8]:
%pip install statsforecast

In [9]:
#!/usr/bin/env python
"""
Audit MuProMAC prefix datasets for internal consistency.

Checks (per file):
- No negative or NaN elapsed_time / remaining_time.
- For each case: elapsed + remaining is constant (cycle time).
- elapsed_time is non-decreasing within a case.
- remaining_time is non-increasing within a case.
- prefix_index sequence per case is 1..n without gaps.
- Every case has at least one COMPLETE event with remaining_time == 0.
"""

import os
import glob

import numpy as np
import pandas as pd

PREFIX_FOLDER = r"out/251110/prefix_datasets"
COMPLETE_STATUS = "COMPLETE"   # change to "complete" if your logs use lowercase


def audit_prefix_file(path: str):
    print(f"\n=== Auditing {os.path.basename(path)} ===")
    df = pd.read_csv(path)

    required_cols = [
        "case_id", "timestamp",
        "elapsed_time", "remaining_time",
        "prefix_index", "status",
        "case_start_time", "case_complete_time",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"  ERROR: missing columns: {missing}")
        return

    print(f"  Rows: {len(df)}, cases: {df['case_id'].nunique()}")

    # ---- 1) NaNs / negatives in time columns ----
    problems = {}

    for col in ["elapsed_time", "remaining_time"]:
        n_nan = df[col].isna().sum()
        n_neg = (df[col] < 0).sum()
        problems[f"{col}_nan"] = n_nan
        problems[f"{col}_neg"] = n_neg

    # ---- 2) elapsed + remaining constant per case ----
    df["cycle_est"] = df["elapsed_time"] + df["remaining_time"]
    cycle_nunique = (
        df.groupby("case_id")["cycle_est"]
          .nunique()
    )
    n_inconsistent_cycle = (cycle_nunique > 1).sum()
    problems["cases_inconsistent_cycle"] = int(n_inconsistent_cycle)

    # ---- 3) elapsed non-decreasing, remaining non-increasing ----
    df_sorted = df.sort_values(["case_id", "timestamp"]).copy()

    diff_elapsed = (
        df_sorted.groupby("case_id")["elapsed_time"]
                 .diff()
    )
    diff_remaining = (
        df_sorted.groupby("case_id")["remaining_time"]
                 .diff()
    )

    # First diff is NaN; treat as okay
    n_elapsed_decrease = (diff_elapsed < 0).sum()
    n_remaining_increase = (diff_remaining > 0).sum()
    problems["rows_elapsed_decrease"] = int(n_elapsed_decrease)
    problems["rows_remaining_increase"] = int(n_remaining_increase)

    # ---- 4) prefix_index sequence 1..n per case ----
    def _prefix_seq_ok(group: pd.DataFrame) -> bool:
        n = len(group)
        expected = np.arange(1, n + 1)
        return np.array_equal(group["prefix_index"].to_numpy(), expected)

    prefix_ok = (
        df_sorted.groupby("case_id")
                 .apply(_prefix_seq_ok)
    )
    n_bad_prefix_cases = (~prefix_ok).sum()
    problems["cases_bad_prefix_index_seq"] = int(n_bad_prefix_cases)

    # ---- 5) COMPLETE rows / remaining_time == 0 ----
    cases_all = set(df["case_id"].unique())
    comp_rows = df[df["status"] == COMPLETE_STATUS]
    cases_with_comp = set(comp_rows["case_id"].unique())
    missing_comp_cases = cases_all - cases_with_comp

    n_missing_comp_cases = len(missing_comp_cases)
    n_comp_nonzero_rem = (comp_rows["remaining_time"] != 0).sum()

    problems["cases_missing_COMPLETE"] = int(n_missing_comp_cases)
    problems["COMP_rows_remaining_not_zero"] = int(n_comp_nonzero_rem)

    # ---- 6) case_start_time and case_complete_time sanity ----
    g = df_sorted.groupby("case_id")

    min_ts = g["timestamp"].min()
    first_start = g["case_start_time"].first()
    start_mismatch_cases = (first_start != min_ts).sum()

    # case_complete_time should be constant per case
    comp_time_nunique = g["case_complete_time"].nunique()
    comp_const_viol = (comp_time_nunique > 1).sum()

    # if a COMPLETE event exists, its timestamp should equal case_complete_time
    if not comp_rows.empty:
        comp_ts = (
            comp_rows.groupby("case_id")["timestamp"]
                     .min()
        )
        # align index with case_complete_time
        common_cases = comp_ts.index.intersection(first_start.index)
        comp_time = g["case_complete_time"].first().loc[common_cases]
        mismatched_complete_ts = (comp_ts.loc[common_cases] != comp_time).sum()
    else:
        mismatched_complete_ts = 0

    problems["cases_start_time_mismatch"] = int(start_mismatch_cases)
    problems["cases_complete_time_not_const"] = int(comp_const_viol)
    problems["cases_complete_ts_mismatch"] = int(mismatched_complete_ts)

    # ---- Summary ----
    any_issue = False
    for k, v in problems.items():
        if v != 0:
            any_issue = True
        print(f"  {k}: {v}")

    if not any_issue:
        print("  ✅ All checks passed for this file.")
    else:
        print("  ⚠️  Some issues detected (see counts above).")


def main():
    if not os.path.isdir(PREFIX_FOLDER):
        raise SystemExit(f"Prefix folder not found: {PREFIX_FOLDER}")

    files = sorted(glob.glob(os.path.join(PREFIX_FOLDER, "*_prefix.csv")))
    if not files:
        raise SystemExit(f"No *_prefix.csv files in {PREFIX_FOLDER}")

    print(f"Found {len(files)} prefix file(s) to audit.")
    for path in files:
        audit_prefix_file(path)


if __name__ == "__main__":
    main()


Found 24 prefix file(s) to audit.

=== Auditing log_FIFO_run0_EXP_dedicated_C1_warm10_test_prefix.csv ===
  Rows: 47885, cases: 1508


C:\Users\990215322\AppData\Local\Temp\ipykernel_22500\1187649084.py:85: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_prefix_seq_ok)


  elapsed_time_nan: 0
  elapsed_time_neg: 0
  remaining_time_nan: 0
  remaining_time_neg: 0
  cases_inconsistent_cycle: 884
  rows_elapsed_decrease: 0
  rows_remaining_increase: 0
  cases_bad_prefix_index_seq: 0
  cases_missing_COMPLETE: 0
  COMP_rows_remaining_not_zero: 0
  cases_start_time_mismatch: 0
  cases_complete_time_not_const: 0
  cases_complete_ts_mismatch: 0
  ⚠️  Some issues detected (see counts above).

=== Auditing log_FIFO_run0_EXP_dedicated_C1_warm10_train_prefix.csv ===
  Rows: 142404, cases: 4491


C:\Users\990215322\AppData\Local\Temp\ipykernel_22500\1187649084.py:85: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_prefix_seq_ok)


  elapsed_time_nan: 0
  elapsed_time_neg: 0
  remaining_time_nan: 0
  remaining_time_neg: 0
  cases_inconsistent_cycle: 2472
  rows_elapsed_decrease: 0
  rows_remaining_increase: 0
  cases_bad_prefix_index_seq: 0
  cases_missing_COMPLETE: 0
  COMP_rows_remaining_not_zero: 0
  cases_start_time_mismatch: 0
  cases_complete_time_not_const: 0
  cases_complete_ts_mismatch: 0
  ⚠️  Some issues detected (see counts above).

=== Auditing log_FIFO_run0_EXP_dedicated_C1_warm10_val_prefix.csv ===
  Rows: 47885, cases: 1508


...

C:\Users\990215322\AppData\Local\Temp\ipykernel_22500\1187649084.py:85: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_prefix_seq_ok)


  elapsed_time_nan: 0
  elapsed_time_neg: 0
  remaining_time_nan: 0
  remaining_time_neg: 0
  cases_inconsistent_cycle: 668
  rows_elapsed_decrease: 0
  rows_remaining_increase: 0
  cases_bad_prefix_index_seq: 0
  cases_missing_COMPLETE: 0
  COMP_rows_remaining_not_zero: 0
  cases_start_time_mismatch: 0
  cases_complete_time_not_const: 0
  cases_complete_ts_mismatch: 0
  ⚠️  Some issues detected (see counts above).

=== Auditing log_FIFO_run0_all_dedicated_sticky_with_rework_warm10_train_prefix.csv ===
  Rows: 103917, cases: 4557


C:\Users\990215322\AppData\Local\Temp\ipykernel_22500\1187649084.py:85: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_prefix_seq_ok)


  elapsed_time_nan: 0
  elapsed_time_neg: 0
  remaining_time_nan: 0
  remaining_time_neg: 0
  cases_inconsistent_cycle: 2320
  rows_elapsed_decrease: 0
  rows_remaining_increase: 0
  cases_bad_prefix_index_seq: 0
  cases_missing_COMPLETE: 0
  COMP_rows_remaining_not_zero: 0
  cases_start_time_mismatch: 0
  cases_complete_time_not_const: 0
  cases_complete_ts_mismatch: 0
  ⚠️  Some issues detected (see counts above).

=== Auditing log_FIFO_run0_all_dedicated_sticky_with_rework_warm10_val_prefix.csv ===
  Rows: 34830, cases: 1530
  elapsed_time_nan: 0
  elapsed_time_neg: 0
  remaining_time_nan: 0
  remaining_time_neg: 0
  cases_inconsistent_cycle: 710
  rows_elapsed_decrease: 0
  rows_remaining_increase: 0
  cases_bad_prefix_index_seq: 0
  cases_missing_COMPLETE: 0
  COMP_rows_remaining_not_zero: 0
  cases_start_time_mismatch: 0
  cases_complete_time_not_const: 0
  cases_complete_ts_mismatch: 0
  ⚠️  Some issues detected (see counts above).


C:\Users\990215322\AppData\Local\Temp\ipykernel_22500\1187649084.py:85: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_prefix_seq_ok)


In [10]:
#!/usr/bin/env python
"""
Check how much elapsed_time + remaining_time varies inside each case.

If differences are tiny (e.g. 1e-6), it's just float rounding noise.
If differences are large, there may be a bug in prefix construction.
"""

import os
import glob

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------
# CONFIG – change this to your prefix folder
# ----------------------------------------------------------------------
PREFIX_FOLDER = r"out/251110/prefix_datasets"
# How many "worst" cases per file to print
TOP_K = 5
# ----------------------------------------------------------------------


def check_file(path: str):
    print(f"\n=== Checking {os.path.basename(path)} ===")
    df = pd.read_csv(path)

    # sanity check: required columns
    for col in ["case_id", "elapsed_time", "remaining_time"]:
        if col not in df.columns:
            print(f"  ERROR: missing column {col}, skipping.")
            return

    # cycle_est = elapsed + remaining
    df["cycle_est"] = df["elapsed_time"] + df["remaining_time"]

    # per-case min/max of cycle_est
    cycle_stats = (
        df.groupby("case_id")["cycle_est"]
          .agg(["min", "max"])
    )
    cycle_stats["diff"] = cycle_stats["max"] - cycle_stats["min"]
    # avoid division by zero
    cycle_stats["rel_diff"] = cycle_stats["diff"] / cycle_stats["max"].replace(0, np.nan)

    # overall numbers
    max_diff = cycle_stats["diff"].max()
    max_rel_diff = cycle_stats["rel_diff"].max()

    print(f"  Cases: {len(cycle_stats)}")
    print(f"  Max absolute diff   (max(cycle_est) - min(cycle_est)): {max_diff}")
    print(f"  Max relative diff   (diff / max(cycle_est)):           {max_rel_diff}")

    # show a few worst cases
    worst = cycle_stats.sort_values("diff", ascending=False).head(TOP_K)
    print("\n  Top cases by absolute diff:")
    for idx, row in worst.iterrows():
        print(
            f"    case_id={idx}: "
            f"min={row['min']}, max={row['max']}, "
            f"diff={row['diff']}, rel_diff={row['rel_diff']}"
        )


def main():
    if not os.path.isdir(PREFIX_FOLDER):
        raise SystemExit(f"Prefix folder not found: {PREFIX_FOLDER}")

    files = sorted(glob.glob(os.path.join(PREFIX_FOLDER, "*_prefix.csv")))
    if not files:
        raise SystemExit(f"No *_prefix.csv files found in {PREFIX_FOLDER}")

    print(f"Found {len(files)} prefix file(s) to check.")

    for path in files:
        check_file(path)


if __name__ == "__main__":
    main()


Found 24 prefix file(s) to check.

=== Checking log_FIFO_run0_EXP_dedicated_C1_warm10_test_prefix.csv ===
  Cases: 1508
  Max absolute diff   (max(cycle_est) - min(cycle_est)): 1.1368683772161603e-13
  Max relative diff   (diff / max(cycle_est)):           4.318301441929393e-16

  Top cases by absolute diff:
    case_id=7372: min=263.2674889662666, max=263.26748896626674, diff=1.1368683772161603e-13, rel_diff=4.318301441929393e-16
    case_id=7355: min=296.34789186349366, max=296.34789186349377, diff=1.1368683772161603e-13, rel_diff=3.836262745340649e-16
    case_id=7390: min=277.22627363448186, max=277.22627363448197, diff=1.1368683772161603e-13, rel_diff=4.1008680826374398e-16
    case_id=7374: min=245.383226148966, max=245.3832261489661, diff=8.526512829121202e-14, rel_diff=3.4747741167707066e-16
    case_id=7395: min=238.69020522816567, max=238.69020522816575, diff=8.526512829121202e-14, rel_diff=3.5722089312255793e-16

=== Checking log_FIFO_run0_EXP_dedicated_C1_warm10_train_prefi

In [ ]:
#!/usr/bin/env python

"""
ETS baseline (statsforecast.AutoETS) for remaining-time prediction.

Assumptions:
- You already ran:
    1) warm-up trimming + train/val/test split
    2) prefix builder

So in `PREFIX_INPUT_FOLDER` you have files like:
    ..._warm10_train_prefix.csv
    ..._warm10_val_prefix.csv
    ..._warm10_test_prefix.csv

For each *base* log, we:
  1) Build a case-level cycle-time series from the TRAIN prefixes.
  2) Fit AutoETS (Hyndman ETS) on that series.
  3) Forecast horizon = (#val cases + #test cases).
  4) Map forecasted total cycle times to val/test cases.
  5) For every prefix row, compute:
        D_hat_ets  = predicted total cycle time for that case
        R_hat_ets  = max(D_hat_ets - elapsed_time, 0)

Outputs:
- One CSV per val/test input, with extra ETS columns, written to OUTPUT_FOLDER.
"""

import os
import glob

import numpy as np
import pandas as pd

from statsforecast import StatsForecast
from statsforecast.models import AutoETS


# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
PREFIX_INPUT_FOLDER = r"out/251110/prefix_datasets"        # where *_prefix.csv live
OUTPUT_FOLDER       = r"out/251110/prefix_with_ets"        # where we'll save predictions

# AutoETS config: no seasonality, automatic error/trend choice
AUTOETS_MODEL      = "ZZN"   # automatic error/trend, no seasonal component
AUTOETS_SEASON_LEN = 1       # no meaningful seasonality in case sequence
# ------------------------------------------------------------------


def load_prefix_file(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    required = [
        "case_id",
        "timestamp",
        "elapsed_time",
        "remaining_time",
        "case_start_time",
        "case_complete_time",
    ]
    for col in required:
        if col not in df.columns:
            raise ValueError(f"{path} missing required column: {col}")

    return df


def build_case_table(df_prefix: pd.DataFrame) -> pd.DataFrame:
    """
    From a prefix dataset, get one row per case with:
    - case_id
    - case_start_time
    - case_complete_time
    - cycle_time = complete - start

    We take the first row per case (they all share the same start/complete times).
    """
    cols = ["case_id", "case_start_time", "case_complete_time"]
    cases = (
        df_prefix
        .sort_values(["case_id", "timestamp"])
        .drop_duplicates(subset=["case_id"], keep="first")
        [cols]
        .copy()
    )
    cases["cycle_time"] = cases["case_complete_time"] - cases["case_start_time"]
    # sort by start time to define the time-series order
    cases = cases.sort_values("case_start_time").reset_index(drop=True)
    return cases


def build_ts_df(cases: pd.DataFrame, unique_id: str) -> pd.DataFrame:
    """
    Build the statsforecast-style time series dataframe for AutoETS:

        columns: [unique_id, ds, y]

    where:
      - unique_id: same string for all rows (each scenario is one series)
      - ds: integer time index (0,1,2,...)
      - y: cycle_time
    """
    n = len(cases)
    ts_df = pd.DataFrame({
        "unique_id": unique_id,
        "ds": np.arange(n),
        "y": cases["cycle_time"].to_numpy(),
    })
    return ts_df


def fit_autoets(train_ts: pd.DataFrame) -> StatsForecast:
    """
    Fit AutoETS on the training time series.
    """
    sf = StatsForecast(
        models=[AutoETS(model=AUTOETS_MODEL, season_length=AUTOETS_SEASON_LEN)],
        freq=1,   # arbitrary integer index frequency
    )
    sf.fit(df=train_ts)
    return sf


def forecast_cycle_times(sf: StatsForecast, h: int) -> np.ndarray:
    """
    Forecast h future cycle times using AutoETS.
    Returns a 1D numpy array of length h.
    """
    y_hat = sf.predict(h=h)
    # column name is the model class name, here "AutoETS"
    preds = y_hat["AutoETS"].to_numpy()
    if len(preds) != h:
        raise RuntimeError(f"Expected {h} forecasts, got {len(preds)}")
    return preds


def add_ets_predictions_to_prefix(
    df_prefix: pd.DataFrame,
    case_table: pd.DataFrame,
    D_hat: np.ndarray,
    case_ids_in_order: np.ndarray,
) -> pd.DataFrame:
    """
    Attach case-level ETS predictions to every prefix row.

    Inputs:
    - df_prefix: prefix-level dataframe (val or test).
    - case_table: dataframe with one row per case.
    - D_hat: array of predicted total cycle times, aligned with case_ids_in_order.
    - case_ids_in_order: np.array of case_ids (sorted) that correspond to D_hat.

    Output:
    - df_prefix copy with:
        * D_hat_ets
        * R_hat_ets = max(D_hat_ets - elapsed_time, 0)
    """
    df_prefix = df_prefix.copy()

    # map from case_id -> predicted total cycle time
    case_pred_df = pd.DataFrame({
        "case_id": case_ids_in_order,
        "D_hat_ets": D_hat,
    })

    # merge predictions into prefix rows
    df_prefix = df_prefix.merge(case_pred_df, on="case_id", how="left")

    # compute predicted remaining time
    df_prefix["R_hat_ets"] = np.clip(
        df_prefix["D_hat_ets"] - df_prefix["elapsed_time"],
        a_min=0.0,
        a_max=None,
    )

    return df_prefix


def process_scenario(train_path: str):
    """
    For one "base" scenario, find its train/val/test prefix files,
    fit AutoETS on train, predict for val+test, and write out new CSVs.
    """
    base_name = os.path.basename(train_path)
    base_root = base_name.replace("_warm10_train_prefix.csv", "")
    print(f"\n=== Scenario: {base_root} ===")

    # infer paths
    val_path  = os.path.join(
        PREFIX_INPUT_FOLDER, f"{base_root}_warm10_val_prefix.csv"
    )
    test_path = os.path.join(
        PREFIX_INPUT_FOLDER, f"{base_root}_warm10_test_prefix.csv"
    )

    if not os.path.exists(val_path) or not os.path.exists(test_path):
        print(f"  Skipping: missing val or test for {base_root}")
        return

    # --- load prefix splits ---
    df_train = load_prefix_file(train_path)
    df_val   = load_prefix_file(val_path)
    df_test  = load_prefix_file(test_path)

    print(f"  Train prefixes: {len(df_train)}, cases: {df_train['case_id'].nunique()}")
    print(f"  Val   prefixes: {len(df_val)}, cases: {df_val['case_id'].nunique()}")
    print(f"  Test  prefixes: {len(df_test)}, cases: {df_test['case_id'].nunique()}")

    # --- build case tables (one row per case) ---
    cases_train = build_case_table(df_train)
    cases_val   = build_case_table(df_val)
    cases_test  = build_case_table(df_test)

    print(f"  Train cases: {len(cases_train)}, "
          f"Val cases: {len(cases_val)}, Test cases: {len(cases_test)}")

    if len(cases_train) == 0:
        print("  No train cases -> skip.")
        return

    # --- build training time series for statsforecast ---
    unique_id = base_root  # any string to identify this series
    ts_train  = build_ts_df(cases_train, unique_id=unique_id)

    # --- fit AutoETS on train series ---
    sf = fit_autoets(ts_train)

    # --- forecast cycle times for val + test cases ---
    h_val  = len(cases_val)
    h_test = len(cases_test)
    h_total = h_val + h_test

    if h_total == 0:
        print("  No val/test cases -> nothing to predict.")
        return

    D_hat_all = forecast_cycle_times(sf, h=h_total)
    D_hat_val = D_hat_all[:h_val]
    D_hat_test = D_hat_all[h_val:]

    # case_ids in the same order as cases_val / cases_test
    case_ids_val  = cases_val["case_id"].to_numpy()
    case_ids_test = cases_test["case_id"].to_numpy()

    # --- attach predictions to prefix rows ---
    df_val_ets = add_ets_predictions_to_prefix(
        df_val, cases_val, D_hat_val, case_ids_val
    )
    df_test_ets = add_ets_predictions_to_prefix(
        df_test, cases_test, D_hat_test, case_ids_test
    )

    # --- save outputs ---
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    out_val  = os.path.join(OUTPUT_FOLDER, f"{base_root}_warm10_val_prefix_ets.csv")
    out_test = os.path.join(OUTPUT_FOLDER, f"{base_root}_warm10_test_prefix_ets.csv")

    df_val_ets.to_csv(out_val, index=False)
    df_test_ets.to_csv(out_test, index=False)

    print(f"  Saved:\n"
          f"    {out_val}  ({len(df_val_ets)} rows)\n"
          f"    {out_test} ({len(df_test_ets)} rows)")


def main():
    if not os.path.isdir(PREFIX_INPUT_FOLDER):
        raise SystemExit(f"Prefix folder not found: {PREFIX_INPUT_FOLDER}")

    # find all *_warm10_train_prefix.csv as "base" scenarios
    train_files = sorted(
        glob.glob(os.path.join(PREFIX_INPUT_FOLDER, "*_warm10_train_prefix.csv"))
    )

    if not train_files:
        raise SystemExit(f"No '*_warm10_train_prefix.csv' files in {PREFIX_INPUT_FOLDER}")

    print(f"Found {len(train_files)} scenario(s) to process.")
    for train_path in train_files:
        process_scenario(train_path)


if __name__ == "__main__":
    main()


In [13]:
#!/usr/bin/env python
"""
Analyze distribution of total cycle times per case for multiple event logs.

For each log file:
- Read CSV event log
- Convert timestamp column to datetime
- For each case, compute cycle time = last_timestamp - first_timestamp
- Print summary statistics
- Save per-case cycle times to CSV
- Save a histogram plot of the cycle time distribution

Adjust the CONFIG section below to match your file paths and column names.
"""

import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


# ----------------------------------------------------------------------
# CONFIG –– EDIT THIS PART
# ----------------------------------------------------------------------

# List 2–3 event log files you want to analyze
LOG_FILES = [
    "out/251110/results/FIFO_EXP_l0.28_dedicated_C1_V6_A5_QC97_identical.csv",
    "out/251110/results/FIFO_EXP_l0.28_dedicated_C1_V6_A5_QC97_mild_all.csv",
    "out/251110/results/FIFO_EXP_l0.28_dedicated_C1_V6_A8_QC97_identical.csv",
    "out/251110/results/FIFO_EXP_l0.28_dedicated_C1_V6_A8_QC97_mild_all.csv",
]

# Output directory for per-case cycle-time CSVs and histograms.
# By default, use the same folder as each input file. You can override here if you want a central folder.
USE_SAME_DIR_AS_INPUT = True
GLOBAL_OUTPUT_DIR = Path("out/251110/cycle_time_analysis")


# ----------------------------------------------------------------------
# CORE LOGIC –– USUALLY NO NEED TO TOUCH
# ----------------------------------------------------------------------

def compute_cycle_times(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a DataFrame with columns:
    - case_id
    - cycle_time

    Strategy:
      1. If 'cycle_time' exists and is populated on COMPLETE events, use that.
      2. Otherwise, compute cycle_time = max(timestamp) - min(timestamp) per case.
    """
    if "case_id" not in df.columns:
        raise ValueError("Input log is missing 'case_id' column.")
    if "timestamp" not in df.columns:
        raise ValueError("Input log is missing 'timestamp' column.")

    # Prefer the explicit cycle_time for COMPLETE events if available
    if "cycle_time" in df.columns:
        df_complete = df[df["status"] == "COMPLETE"].copy()
        # Some safety: drop NaNs in cycle_time
        df_complete = df_complete.dropna(subset=["cycle_time"])

        if not df_complete.empty:
            # Expect 1 COMPLETE per case; if multiple, take the last one
            df_complete = (
                df_complete.sort_values(["case_id", "timestamp"])
                           .groupby("case_id", as_index=False)
                           .tail(1)
            )
            return df_complete[["case_id", "cycle_time"]].reset_index(drop=True)

    # Fallback: compute from timestamp range per case
    grouped = df.groupby("case_id")["timestamp"].agg(["min", "max"]).reset_index()
    grouped["cycle_time"] = grouped["max"] - grouped["min"]
    return grouped[["case_id", "cycle_time"]]


def analyze_single_log(log_path: Path, output_dir: Path):
    print(f"\n=== Analyzing log: {log_path} ===")
    if not log_path.exists():
        print(f"  [WARNING] File not found, skipping: {log_path}")
        return

    # Make sure output dir exists
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load log
    df = pd.read_csv(log_path)

    # Compute per-case cycle times
    ct_df = compute_cycle_times(df)

    # Basic stats
    desc = ct_df["cycle_time"].describe()
    print("  #cases:", len(ct_df))
    print("  cycle_time stats (simulation time units):")
    print(desc.to_string())

    # Save per-case cycle times to CSV
    base = log_path.stem
    out_csv = output_dir / f"{base}_cycle_times_per_case.csv"
    ct_df.to_csv(out_csv, index=False)
    print(f"  Saved per-case cycle times to: {out_csv}")

    # Plot histogram
    plt.figure()
    plt.hist(ct_df["cycle_time"], bins=50)
    plt.xlabel("Cycle time (same time units as 'timestamp')")
    plt.ylabel("Number of cases")
    plt.title(f"Cycle time distribution\n{base}")
    plt.tight_layout()

    out_png = output_dir / f"{base}_cycle_times_hist.png"
    plt.savefig(out_png, dpi=150)
    plt.close()
    print(f"  Saved histogram to: {out_png}")


def main():
    log_paths = [Path(p) for p in LOG_FILES]

    for log_path in log_paths:
        if USE_SAME_DIR_AS_INPUT:
            out_dir = log_path.parent / "cycle_time_analysis"
        else:
            out_dir = GLOBAL_OUTPUT_DIR

        analyze_single_log(log_path, out_dir)


if __name__ == "__main__":
    main()


=== Analyzing log: out\251110\results\FIFO_EXP_l0.28_dedicated_C1_V6_A5_QC97_identical.csv ===
  #cases: 8487
  cycle_time stats (simulation time units):
count    8487.000000
mean      101.735590
std        53.928117
min         5.103604
25%        61.360799
50%        91.433152
75%       132.428979
max       433.400226
  Saved per-case cycle times to: out\251110\results\cycle_time_analysis\FIFO_EXP_l0.28_dedicated_C1_V6_A5_QC97_identical_cycle_times_per_case.csv
  Saved histogram to: out\251110\results\cycle_time_analysis\FIFO_EXP_l0.28_dedicated_C1_V6_A5_QC97_identical_cycle_times_hist.png

=== Analyzing log: out\251110\results\FIFO_EXP_l0.28_dedicated_C1_V6_A5_QC97_mild_all.csv ===
  #cases: 8312
  cycle_time stats (simulation time units):
count    8312.000000
mean       89.804345
std        49.880785
min         9.677472
25%        53.815093
50%        78.606236
75%       113.628093
max       545.547747
  Saved per-case cycle times to: out\251110\results\cycle_time_analysis\FIFO_E

In [ ]:
#!/usr/bin/env python
"""
Plot distribution of per-case cycle times for one or more MuProMAC/FIFO logs.

Definition used here:
    cycle_time(case) = max(timestamp) - min(timestamp)
where 'timestamp' is the simulation time in minutes written by the simulator.

We ignore the 'cycle_time' column in the CSV and recompute from timestamps.
"""

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# ----------------------------------------------------------------------
# CONFIG – edit this list
# ----------------------------------------------------------------------

LOG_FILES = [
    "out/251110/results/FIFO_EXP_l0.28_dedicated_C1_V6_A5_QC97_identical.csv",
    "out/251110/results/FIFO_EXP_l0.28_dedicated_C1_V6_A8_QC97_mild_all.csv",
]

BINS = 40  # number of histogram bins

# ----------------------------------------------------------------------
# CORE
# ----------------------------------------------------------------------

def get_cycle_times_from_timestamps(path: Path) -> pd.Series:
    """
    Compute cycle time per case from timestamp range.
    Returns a pandas Series of cycle times (float, minutes).
    """
    df = pd.read_csv(path)

    # Make absolutely sure 'timestamp' is numeric (no datetime nonsense)
    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")

    # Group by case and compute max(timestamp) - min(timestamp)
    rng = df.groupby("case_id")["timestamp"].agg(["min", "max"])
    ct = (rng["max"] - rng["min"]).astype(float)

    return ct


def plot_cycle_time_distribution(path: Path, bins: int = 40):
    ct = get_cycle_times_from_timestamps(path)

    print(f"\n=== {path} ===")
    print(f"#cases: {len(ct)}")
    print(ct.describe())

    plt.figure(figsize=(6, 4))
    plt.hist(ct.values, bins=bins)
    plt.xlabel("Cycle time (minutes)")
    plt.ylabel("Number of cases")
    plt.title(f"Cycle time distribution (minutes)\n{path.name}")

    # Kill scientific notation / offsets on x-axis
    ax = plt.gca()
    ax.ticklabel_format(style="plain", axis="x", useOffset=False)
    ax.get_xaxis().get_offset_text().set_visible(False)

    plt.tight_layout()

    out_png = path.with_name(path.stem + "_cycle_time_hist.png")
    plt.savefig(out_png, dpi=150)
    plt.close()

    print(f"Saved histogram to: {out_png}")


def main():
    for p in LOG_FILES:
        plot_cycle_time_distribution(Path(p), bins=BINS)


if __name__ == "__main__":
    main()



=== out\251110\results\FIFO_EXP_l0.28_dedicated_C1_V6_A5_QC97_identical.csv ===
#cases: 8521
count    8521.000000
mean      101.517153
std        54.046708
min         0.000000
25%        61.230344
50%        91.308206
75%       132.411357
max       433.400226
dtype: float64
Saved histogram to: out\251110\results\FIFO_EXP_l0.28_dedicated_C1_V6_A5_QC97_identical_cycle_time_hist.png

=== out\251110\results\FIFO_EXP_l0.28_dedicated_C1_V6_A8_QC97_mild_all.csv ===
#cases: 8446
count    8446.000000
mean      106.536554
std        58.829877
min         0.000000
25%        62.414378
50%        93.764079
75%       138.144935
max       560.701494
dtype: float64
Saved histogram to: out\251110\results\FIFO_EXP_l0.28_dedicated_C1_V6_A8_QC97_mild_all_cycle_time_hist.png
